# M2 — Parse một Range và benchmark Colab CPU

**Cách chạy:** chọn `MEMBER` ở ô dưới → **Runtime → Run all**. Mỗi người dùng cùng
notebook, Range tự đọc từ `configs/ranges.toml` (kể cả Range qua hai tháng).

Trong biểu tượng 🔑 **Secrets**, bật quyền notebook cho token đọc dataset, tên
`HF_TOKEN` (cũng nhận `HF_API_KEY`, `HUGGINGFACE_TOKEN`, `HUGGING_FACE_HUB_TOKEN`).
Không dán token vào code. M2 không cần key Semantic Scholar hay email liên hệ.

Notebook tải **source và metadata M1** của người được chọn từ HF, kiểm checksum,
rồi parse offline bằng các gói thật. Thời gian tải/cài đặt được ghi riêng với
thời gian benchmark Parser. Không crawl lại arXiv hoặc tự upload lên HF.

Kết quả: ba bảng Parquet, log từng Version, danh sách ID thiếu dữ liệu, metrics,
benchmark RAM/đĩa, thông tin CPU và phiên bản dependency, checkpoint để resume.
Chỉ chọn **CPU** trong Change runtime type; không cần GPU.

In [ ]:
# @title 1. Chọn thành viên (chỉ cần đổi MEMBER)
MEMBER = "minh" # @param ["bao", "minh", "nghiem", "hai", "lieu"]

# Các giá trị bên dưới có thể giữ nguyên.
HF_REPO_ID = "hcmus-arxiv-reference/arXiv-reference-matching"
CODE_REF = "a800037166da8feff2fbb739a7107e4edb615ad1"  # main sau PR #60
HF_REVISION = "main"  # Phân giải thành SHA, cố định trong suốt lượt chạy.
LIMIT = 0 # @param {type:"integer"}
# LIMIT=0: toàn Range; LIMIT=3: kiểm tra nhanh 3 ID đầu có dữ liệu trên HF.
BATCH_SIZE = 50
SAVE_TO_DRIVE = False # @param {type:"boolean"}
# True: thêm bản sao checkpoint/kết quả vào MyDrive/arxiv-m2 sau mỗi lô.
# False: không cần mount Drive; cuối lượt có nút tải kết quả về máy.

In [ ]:
# @title 2. Lấy code đã chốt và cài dependency vào runtime Colab
import os
import sys
import json
import subprocess
import time
from pathlib import Path

SETUP_STARTED = time.perf_counter()
if sys.version_info < (3, 12):
    raise RuntimeError("Cần Python >=3.12. Hãy dùng runtime Colab mới.")
REPO = Path("/content/arxiv-m2-code") / CODE_REF[:12]
REPO.parent.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout",
                    "https://github.com/hcmus-arxiv-reference/arxiv-reference-matching.git",
                    str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", CODE_REF], check=True)
CODE_SHA = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r",
                str(REPO / "requirements.txt")], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
SETUP_SECONDS = time.perf_counter() - SETUP_STARTED
print("Python:", sys.version.split()[0], "| code:", CODE_SHA)
print("Cài đặt xong; chưa bắt đầu benchmark.")

In [ ]:
from pathlib import Path
import shutil
from collections import Counter

from src.common.config import load_ranges, member_ids
from src.common.corpus import FileCorpus, file_digest, verified_source
from src.common.json_io import write_json
from src.common.schema import publication_key


def choose_hf_member(api, repo_id, revision, member):
    """Resolve only the selected member's folder; never substitute another Range."""
    ranges = load_ranges()
    folders = [entry.path.rsplit("/", 1)[-1] for entry in api.list_repo_tree(
        repo_id, repo_type="dataset", revision=revision, path_in_repo="raw", recursive=False)
        if type(entry).__name__ == "RepoFolder"]
    aliases = {member.casefold(), ranges[member].github.casefold()}
    matches = [name for name in folders if name.casefold() in aliases]
    if len(matches) != 1:
        raise ValueError(f"Không xác định được thư mục HF cho {member}. Các thư mục: {folders}")
    return matches[0]


def hf_publication_keys(api, repo_id, revision, folder):
    return {entry.path.rsplit("/", 1)[-1] for entry in api.list_repo_tree(
        repo_id, repo_type="dataset", revision=revision,
        path_in_repo=f"raw/{folder}", recursive=False)
        if type(entry).__name__ == "RepoFolder"}


def download_inputs(api, repo_id, revision, folder, ids, stage, token):
    """List only selected subtrees; download in bounded groups without scanning other members."""
    from concurrent.futures import ThreadPoolExecutor
    from itertools import islice
    from huggingface_hub import hf_hub_download

    keys = {publication_key(aid) for aid in ids}
    roots = [f"raw/{folder}"] if len(keys) > 20 else [f"raw/{folder}/{key}" for key in sorted(keys)]

    def filenames():
        for root in roots:
            for entry in api.list_repo_tree(repo_id, repo_type="dataset", revision=revision,
                                            path_in_repo=root, recursive=True):
                if (type(entry).__name__ == "RepoFile" and entry.path.split("/")[2] in keys
                        and (Path(entry.path).name in {"source.json", "metadata.json"}
                             or Path(entry.path).suffix.lower() in {".tex", ".bib"})):
                    yield entry.path

    def download(filename):
        hf_hub_download(repo_id, filename, repo_type="dataset", revision=revision,
                        local_dir=stage, token=token)

    iterator = iter(filenames())
    count = 0
    print(f"Đang tải source của {len(ids)} Publication từ raw/{folder}/...", flush=True)
    with ThreadPoolExecutor(max_workers=4) as executor:
        while batch := list(islice(iterator, 250)):
            for _ in executor.map(download, batch):
                count += 1
            print(f"Đã tải/kiểm cache {count} file.", flush=True)
    return count


def prepare_corpus(ids, corpus_dir, hf_root):
    """Keep missing inputs in the audit; reject corrupt inputs without editing raw."""
    corpus_dir = Path(corpus_dir)
    hf_root = Path(hf_root)
    corpus = FileCorpus(corpus_dir)
    report = []
    for aid in ids:
        key = publication_key(aid)
        destination = corpus_dir / "raw" / key
        staged = hf_root / key
        if staged.exists() and not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(staged), str(destination))
        metadata = corpus.metadata_path(aid)
        original = destination / "v1" / "metadata.json"
        if not metadata.exists() and original.exists():
            metadata.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(original, metadata)
        if not metadata.exists():
            report.append({"arxiv_id": aid, "status": "missing_metadata", "versions": None})
            continue
        publication = corpus.read_publication(aid)
        missing = []
        states = Counter()
        for version in range(1, publication.versions + 1):
            directory = corpus.version_path(aid, version)
            if not (directory / "source.json").exists():
                missing.append(version)
                continue
            source = verified_source(directory)
            if source is None:
                raise ValueError(f"{aid}v{version}: checksum/receipt không hợp lệ; dừng để kiểm tra dữ liệu.")
            replica = directory / "metadata.json"
            if replica.exists() and file_digest(replica) != file_digest(metadata):
                raise ValueError(f"{aid}v{version}: metadata giữa các Version không khớp.")
            states[source.status] += 1
        report.append({"arxiv_id": aid, "status": "missing_source" if missing else "verified",
                       "versions": publication.versions, "missing_versions": missing,
                       "source_statuses": dict(states)})
    return report

In [ ]:
# @title 3. Đọc Secret và kiểm tra dữ liệu đúng Range
from google.colab import userdata
from huggingface_hub import HfApi

RANGES = load_ranges()
if MEMBER not in RANGES or LIMIT < 0 or BATCH_SIZE < 1:
    raise ValueError("MEMBER, LIMIT hoặc BATCH_SIZE không hợp lệ.")
HF_TOKEN = None
for secret_name in ("HF_TOKEN", "HF_API_KEY", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
    try:
        HF_TOKEN = userdata.get(secret_name)
    except userdata.SecretNotFoundError:
        continue
    if HF_TOKEN:
        print("Đã đọc Secret:", secret_name, "(giá trị được giữ kín)")
        break
if not HF_TOKEN:
    raise RuntimeError("Thêm HF_TOKEN vào Colab Secrets và bật Notebook access.")
API = HfApi(token=HF_TOKEN)
try:
    HF_SHA = API.repo_info(HF_REPO_ID, repo_type="dataset", revision=HF_REVISION).sha
    HF_MEMBER = choose_hf_member(API, HF_REPO_ID, HF_SHA, MEMBER)
    AVAILABLE_KEYS = hf_publication_keys(API, HF_REPO_ID, HF_SHA, HF_MEMBER)
except Exception as error:
    raise RuntimeError(f"Không đọc được inventory HF ({type(error).__name__}). "
                       "Kiểm tra quyền đọc dataset của Secret; không cần gửi token.") from None

RANGE_IDS = list(member_ids(MEMBER))
TARGET_IDS = RANGE_IDS if LIMIT == 0 else [
    aid for aid in RANGE_IDS if publication_key(aid) in AVAILABLE_KEYS][:LIMIT]
if not TARGET_IDS:
    raise RuntimeError(f"HF không có dữ liệu cho Range {MEMBER}.")
RUN_KEY = f"{MEMBER}-{CODE_SHA[:8]}-{HF_SHA[:8]}-n{len(TARGET_IDS)}-b{BATCH_SIZE}"
WORK = Path("/content/arxiv-m2") / RUN_KEY
CORPUS = WORK / "corpus"
STAGE = WORK / "hf-download"
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Thành viên:", MEMBER, "| GitHub:", RANGES[MEMBER].github)
print("Range:", RANGE_IDS[0], "→", RANGE_IDS[-1], "|", len(RANGE_IDS), "ID")
print("Chế độ:", "TOÀN RANGE" if LIMIT == 0 else "KIỂM TRA NHANH", "|", len(TARGET_IDS), "ID")
print("Nguồn:", f"raw/{HF_MEMBER}/", "| HF SHA:", HF_SHA)
print("ID có thư mục HF:", sum(publication_key(aid) in AVAILABLE_KEYS for aid in TARGET_IDS),
      "/", len(TARGET_IDS), "| ID thiếu vẫn được giữ trong input-report.json.")
print("Thư mục kết quả:", RESULTS)

In [ ]:
# @title 4. Tải dữ liệu và kiểm checksum (ngoài thời gian benchmark)
DOWNLOAD_STARTED = time.perf_counter()
NEEDED = [aid for aid in TARGET_IDS if publication_key(aid) in AVAILABLE_KEYS
          and not (CORPUS / "raw" / publication_key(aid)).exists()]
if NEEDED:
    try:
        download_inputs(API, HF_REPO_ID, HF_SHA, HF_MEMBER, NEEDED, STAGE, HF_TOKEN)
    except Exception as error:
        raise RuntimeError(f"Tải HF bị ngắt ({type(error).__name__}). Chạy lại ô này để tiếp tục.") from None
INPUT_REPORT = prepare_corpus(TARGET_IDS, CORPUS, STAGE / "raw" / HF_MEMBER)
DOWNLOAD_SECONDS = time.perf_counter() - DOWNLOAD_STARTED
write_json(RESULTS / "input-report.json", {
    "member": MEMBER, "hf_repo_id": HF_REPO_ID, "hf_revision": HF_SHA,
    "range_ids": len(RANGE_IDS), "requested_ids": len(TARGET_IDS),
    "download_and_verification_seconds": DOWNLOAD_SECONDS, "publications": INPUT_REPORT})
PARSE_IDS = [row["arxiv_id"] for row in INPUT_REPORT if row["versions"] is not None]
if not PARSE_IDS:
    raise RuntimeError("Không có metadata để parse. Xem input-report.json; không bắt đầu benchmark.")
print("Kiểm đầu vào:", dict(Counter(row["status"] for row in INPUT_REPORT)))
print("Publication có metadata:", len(PARSE_IDS), "/", len(TARGET_IDS))
print("Version theo metadata:", sum(row["versions"] or 0 for row in INPUT_REPORT))
print("Tải + xác minh:", round(DOWNLOAD_SECONDS, 2), "giây; chưa bắt đầu benchmark.")

In [ ]:
# @title 5. Bản sao trên Drive (tùy chọn)
DRIVE_RESULTS = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/arxiv-m2") / RUN_KEY
    if (DRIVE_RESULTS / "results").exists():
        shutil.copytree(DRIVE_RESULTS / "results", RESULTS, dirs_exist_ok=True)
    print("Sau mỗi lô sẽ sao lưu checkpoint vào:", DRIVE_RESULTS)
else:
    print("Đang lưu trên /content. Ngắt kết nối tạm: chạy lại ô benchmark để resume.")
    print("Nếu Colab xóa runtime, cần bản sao Drive hoặc tải checkpoint trước khi thoát.")

In [ ]:
# @title 6. Bộ chạy theo lô và ghép bảng (giữ nguyên schema Parser)
WORKER_SOURCE = r"""
import gc
import json
import os
from pathlib import Path
import shutil
import socket
import sys
import time

import pyarrow.parquet as pq


def merge_outputs(batch_dirs, output):
    # Dedup within each Publication is already done; batches contain disjoint IDs.
    from src.common.schema import SCHEMAS, validate
    tables = output / "tables"
    tables.mkdir(exist_ok=True)
    for name in ("elements", "hierarchy", "bibtex"):
        temporary = tables / f".{name}.parquet"
        with pq.ParquetWriter(temporary, SCHEMAS[name], compression="snappy") as writer:
            for directory in batch_dirs:
                # A full batch remains small; validate uniqueness within the batch.
                table = pq.read_table(directory / "tables" / f"{name}.parquet")
                errors = validate(table, name)
                if errors:
                    raise ValueError(str(errors))
                writer.write_table(table)
        os.replace(temporary, tables / f"{name}.parquet")
    logs = output / "log"
    logs.mkdir(exist_ok=True)
    temporary = logs / ".versions.jsonl"
    with temporary.open("wb") as target:
        for directory in batch_dirs:
            with (directory / "log" / "versions.jsonl").open("rb") as source:
                shutil.copyfileobj(source, target)
    os.replace(temporary, logs / "versions.jsonl")


def run(config):
    # Network is blocked in main; every input is already local.
    from src.common.json_io import write_json
    from src.parse.pipeline import get_module_versions, run_batch
    versions = get_module_versions()
    if any(value.startswith("fallback-") for value in versions.values()):
        raise RuntimeError("Benchmark cần các gói thật, không dùng fallback.")
    output = Path(config["output"])
    output.mkdir(parents=True, exist_ok=True)
    ids = sorted(config["ids"])
    if len(ids) != len(set(ids)):
        raise ValueError("Danh sách Publication bị trùng.")
    started = time.perf_counter()
    metrics = []
    batch_dirs = []
    backup_seconds = 0.0
    for start in range(0, len(ids), config["batch_size"]):
        directory = output / "batches" / f"{start // config['batch_size']:04d}"
        batch_ids = ids[start:start + config["batch_size"]]
        result = run_batch(config["corpus"], directory, arxiv_ids=batch_ids, resume=True)
        if result["fallback_modules"]:
            raise RuntimeError("Pipeline kích hoạt fallback.")
        metrics.append(result)
        batch_dirs.append(directory)
        print(f"Đã xử lý {min(start + len(batch_ids), len(ids))}/{len(ids)} Publication; "
              f"lô này: {result['publications_succeeded']} thành công, "
              f"{result['publications_failed']} lỗi, {result['publications_resumed']} dùng checkpoint.", flush=True)
        gc.collect()
        if config.get("drive_results"):
            t0 = time.perf_counter()
            backup = Path(config["drive_results"]) / "results"
            shutil.copytree(directory, backup / "batches" / directory.name, dirs_exist_ok=True)
            shutil.copy2(output / "input-report.json", backup / "input-report.json")
            backup_seconds += time.perf_counter() - t0
    merge_outputs(batch_dirs, output)
    additive = (
        "publications_total", "publications_succeeded", "publications_empty_only", "publications_failed",
        "publications_resumed", "versions_total_metadata", "versions_parsed", "versions_empty_non_tex",
        "versions_empty_unavailable", "versions_failed", "versions_missing", "elements_before_dedup",
        "elements_after_dedup", "bibtex_entries_before_dedup", "bibtex_entries_after_dedup",
        "bibtex_unresolved_keys", "hierarchy_edges", "duplicate_sibling_edges")
    summary = {key: sum(m.get(key, 0) for m in metrics) for key in additive}
    total = summary["publications_total"]
    resumed = summary["publications_resumed"]
    summary.update({
        "member": config["member"], "code_sha": config["code_sha"], "hf_revision": config["hf_sha"],
        "module_versions": versions, "range_ids": config["range_ids"],
        "requested_ids": config["requested_ids"], "metadata_missing_ids": config["requested_ids"] - total,
        "run_kind": "fresh" if resumed == 0 else ("resume_only" if resumed == total else "mixed_resume"),
        "test_run": config["test_run"], "batch_size": config["batch_size"],
        "worker_wall_seconds": time.perf_counter() - started, "drive_backup_seconds": backup_seconds,
        "batch_pipeline_seconds": sum(m["elapsed_seconds"] for m in metrics),
        "publication_parse_rate": summary["publications_succeeded"] / total if total else 0,
        "range_success_rate": summary["publications_succeeded"] / config["requested_ids"],
        "tables_dir_size_mb": sum(p.stat().st_size for p in (output / "tables").glob("*.parquet")) / 1024**2,
    })
    covered = summary["versions_parsed"] + summary["versions_empty_non_tex"] + summary["versions_empty_unavailable"]
    summary["version_coverage_rate"] = covered / summary["versions_total_metadata"] if summary["versions_total_metadata"] else 0
    for prefix in ("elements", "bibtex_entries"):
        before, after = summary[f"{prefix}_before_dedup"], summary[f"{prefix}_after_dedup"]
        summary[f"{prefix}_dedup_rate"] = 1 - after / before if before else 0
    with (output / "log" / "versions.jsonl").open() as stream:
        rows = sum(1 for line in stream if line.strip())
    if rows != summary["versions_total_metadata"]:
        raise ValueError("Số dòng log khác số Version theo metadata.")
    write_json(Path(config["segment"]) / "metrics.json", summary)
    write_json(output / "metrics.json", summary)
    return summary


if __name__ == "__main__":
    config = json.loads(Path(sys.argv[1]).read_text())
    sys.path.insert(0, config["repo"])
    # Parser reads local files only. A mistaken network request fails immediately.
    def no_network(*args, **kwargs):
        raise RuntimeError("Mạng bị chặn trong benchmark Parser M2.")
    socket.create_connection = no_network
    socket.socket.connect = no_network
    socket.socket.connect_ex = no_network
    run(config)
"""
WORKER = WORK / "m2_worker.py"
WORKER.write_text(WORKER_SOURCE, encoding="utf-8")
print("Bộ chạy đã sẵn sàng; mỗi lô tối đa", BATCH_SIZE, "Publication.")

In [ ]:
# @title 7. Chạy benchmark (chạy lại ô này để resume trong cùng runtime)
import platform
import importlib.metadata
import hashlib

BENCH_ROOT = RESULTS / "benchmarks"
BENCH_ROOT.mkdir(exist_ok=True)
SEGMENT = BENCH_ROOT / time.strftime("segment-%Y%m%d-%H%M%S", time.gmtime())
SEGMENT.mkdir(exist_ok=False)
CONFIG = {
    "repo": str(REPO), "corpus": str(CORPUS), "output": str(RESULTS), "segment": str(SEGMENT),
    "ids": PARSE_IDS, "member": MEMBER, "code_sha": CODE_SHA, "hf_sha": HF_SHA,
    "range_ids": len(RANGE_IDS), "requested_ids": len(TARGET_IDS), "batch_size": BATCH_SIZE,
    "test_run": LIMIT != 0, "drive_results": str(DRIVE_RESULTS) if DRIVE_RESULTS else None,
}
write_json(SEGMENT / "run-config.json", CONFIG)
cpu = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
            if line.startswith("model name")), platform.processor())
write_json(SEGMENT / "environment.json", {
    "python": sys.version, "platform": platform.platform(), "cpu_model": cpu,
    "logical_cpus": os.cpu_count(), "ram_total_mb": __import__("psutil").virtual_memory().total / 1024**2,
    "code_sha": CODE_SHA, "hf_revision": HF_SHA, "member": MEMBER,
    "worker_sha256": hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("pyarrow", "huggingface_hub", "psutil", "bibtexparser")},
    "input_download_and_verification_seconds": DOWNLOAD_SECONDS,
    "setup_seconds": SETUP_SECONDS,
})
# Không truyền credentials vào tiến trình Parser hoặc ghi chúng trong config.
child_env = os.environ.copy()
for key in ("HF_TOKEN", "HF_API_KEY", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN", "S2_API_KEY"):
    child_env.pop(key, None)
command = [sys.executable, "-m", "src.benchmark", "--data-dir", str(RESULTS),
           "--out", str(SEGMENT), "--interval", "1", "--disk-interval", "5", "--",
           sys.executable, str(WORKER), str(SEGMENT / "run-config.json")]
print("Bắt đầu đo Parser. Theo dõi tiến độ mỗi lô phía dưới.")
completed = subprocess.run(command, cwd=REPO, env=child_env)
if completed.returncode:
    raise RuntimeError("Lượt chạy bị ngắt hoặc lỗi. Checkpoint đã xong được giữ lại; chạy lại ô này để resume.")
print("Hoàn thành. Kết quả benchmark:", SEGMENT)

In [ ]:
# @title 8. Đọc kết quả, phân biệt lượt mới và resume
METRICS = json.loads((SEGMENT / "metrics.json").read_text())
BENCHMARK = json.loads((SEGMENT / "benchmark.json").read_text())
if BENCHMARK["return_code"] != 0 or BENCHMARK["interrupted"]:
    raise RuntimeError("Benchmark chưa hoàn thành.")
wall = BENCHMARK["wall_seconds"]
METRICS.update({
    "benchmark_wall_seconds": wall,
    "avg_seconds_per_publication": wall / METRICS["publications_total"],
    "peak_rss_mb": BENCHMARK["peak_rss_mb"], "avg_rss_mb": BENCHMARK["avg_rss_mb"],
    "peak_disk_mb": BENCHMARK["peak_disk_mb"], "final_disk_mb": BENCHMARK["final_disk_mb"],
})
write_json(SEGMENT / "metrics.json", METRICS)
write_json(RESULTS / "metrics.json", METRICS)
print("Chế độ:", "KIỂM TRA NHANH" if METRICS["test_run"] else "TOÀN RANGE", "|", METRICS["run_kind"])
print(f"Publication: {METRICS['publications_total']}/{METRICS['requested_ids']} có metadata; "
      f"{METRICS['publications_succeeded']} thành công; {METRICS['publications_failed']} lỗi.")
print(f"Tỉ lệ thành công trên tất cả ID yêu cầu: {METRICS['range_success_rate']:.2%}")
print(f"Độ phủ Version (chỉ các bài có metadata): {METRICS['version_coverage_rate']:.2%}")
print(f"Thời gian: {wall:.2f}s; trung bình {METRICS['avg_seconds_per_publication']:.3f}s/Publication có metadata.")
print("RAM đỉnh/trung bình (MB):", METRICS["peak_rss_mb"], "/", METRICS["avg_rss_mb"])
print("Ba bảng đầu ra (MB):", round(METRICS["tables_dir_size_mb"], 2))
print("Đĩa đỉnh/cuối thư mục results gồm checkpoint, bảng lô và bảng cuối (MB):",
      METRICS["peak_disk_mb"], "/", METRICS["final_disk_mb"])
print("Element trước/sau:", METRICS["elements_before_dedup"], "/", METRICS["elements_after_dedup"])
print("BibTeX entry trước/sau:", METRICS["bibtex_entries_before_dedup"], "/", METRICS["bibtex_entries_after_dedup"])
if METRICS["run_kind"] != "fresh":
    print("Lượt này có resume: không dùng thời gian này thay cho benchmark parse mới toàn Range.")
if DRIVE_RESULTS:
    shutil.copytree(RESULTS, DRIVE_RESULTS / "results", dirs_exist_ok=True)
    print("Đã sao lưu kết quả + checkpoint trên Drive.")

In [ ]:
# @title 9. Tạo gói tải về (không chứa raw hoặc token)
ARCHIVE = Path(shutil.make_archive(str(WORK / f"{RUN_KEY}-results"), "zip", RESULTS))
print("Gói gồm ba bảng cuối, bảng theo lô, checkpoint, log và số liệu:", ARCHIVE.name)
print("SHA-256:", file_digest(ARCHIVE))
from google.colab import files
files.download(str(ARCHIVE))

## Cách đọc số liệu và chạy tiếp

- `LIMIT=0` yêu cầu mọi ID trong Range. ID thiếu metadata vẫn ở `input-report.json`,
  được tính vào mẫu số `range_success_rate`; không tự bịa số Version cho ID đó.
  Version `non_tex`/`unavailable` giữ bản ghi rỗng đúng hợp đồng.
- `metrics.json` là số liệu toàn lượt; `benchmarks/segment-.../` giữ số liệu riêng
  của từng lần chạy. Lượt có checkpoint được đánh dấu `mixed_resume`/`resume_only`.
  Muốn đo mới độc lập, dùng runtime mới và giữ `SAVE_TO_DRIVE=False`.
- RAM: `src.benchmark` lấy mẫu mỗi giây cho tiến trình chạy và tiến trình con.
  Đĩa: lấy mẫu mỗi 5 giây cho **results**, gồm cả checkpoint và bảng từng lô;
  `tables_dir_size_mb` chỉ tính ba bảng cuối. Các đỉnh là đỉnh **ghi nhận qua lấy mẫu**.
- `worker_wall_seconds` và benchmark wall time gồm ghép/kiểm tra bảng và thời gian
  sao lưu Drive (nếu bật); `drive_backup_seconds` ghi riêng phần sao lưu từng lô.
  `done_papers_*` của harness cũ dành cho crawler; dùng số đếm Publication ở metrics M2.
- Thiếu input, parser thất bại hoặc key thiếu entry được báo trong log; không đồng
  nghĩa notebook hỏng. Benchmark không thay thế nghiệm thu chất lượng bằng nhãn tay.
- Drive bật: phiên mới chọn đúng MEMBER và cấu hình, Run all để tải lại input cùng
  HF revision rồi khôi phục checkpoint. Muốn giữ đúng snapshot sau khi HF đổi,
  điền SHA ở `HF_REVISION` từ `environment.json`/`input-report.json` của lượt cũ.
  Không có Drive: tải gói ZIP để giữ kết quả; checkpoint chỉ tự resume trong runtime hiện tại.

Nguồn: [phân công M2 §4.4](https://github.com/hcmus-arxiv-reference/arxiv-reference-matching/blob/main/docs/m2-cong-viec.md#44-minh-integration-m25-chạy-batch-số-liệu-báo-cáo),
[đề Parsing](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/),
[tải snapshot HF](https://huggingface.co/docs/huggingface_hub/en/guides/download),
[Colab Secrets](https://colab.research.google.com/notebooks/snippets/secrets.ipynb).